# 02 Strict Test-Only Table Model Forecasting

This notebook trains one final model per ticker/model family/horizon under the strict protocol. Hyperparameters are tuned only inside the train block, model families are ranked on validation, and all final artifacts for `03_model_comparison.ipynb` are based on test predictions only. Mature tickers use a rolling maximum of the latest 5 trading years before validation/test fitting; shorter-history tickers use all available history and remain flagged.

In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
for candidate in [cwd, cwd / "forecasting", cwd.parent, cwd.parent / "forecasting"]:
    if (candidate / "src" / "stock_forecast").exists():
        PROJECT_DIR = candidate
        break
else:
    raise RuntimeError("Cannot locate forecasting project directory with src/stock_forecast")

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ARTIFACT_DIR = PROJECT_DIR / "artifacts"
DATA_DIR = ARTIFACT_DIR / "data"
REPORTS_DIR = ARTIFACT_DIR / "reports"
PLOTS_DIR = ARTIFACT_DIR / "plots"
for path in [DATA_DIR, REPORTS_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR = {PROJECT_DIR}")

PROJECT_DIR = /home/sapce/forecasting_stock_prices/forecasting


In [2]:
import importlib.util
import pandas as pd
from IPython.display import display

from stock_forecast.artifacts import load_json, load_table
from stock_forecast.mlflow_tracking import MLflowRunConfig, log_strict_protocol_result, mlflow_horizon_run
from stock_forecast.models import build_model
from stock_forecast.strict_protocol import run_strict_per_ticker_protocol

pd.set_option("display.max_columns", 160)


## Notebook Constants

In [3]:
FORCE_RETRAIN = False
PRIMARY_METRIC = "directional_accuracy"
RANDOM_STATE = 42
N_TRIALS = 25
OPTUNA_N_JOBS = 2

STRICT_VALIDATION_ROWS = 126
STRICT_TEST_ROWS = 126
MATURE_MIN_ROWS = 1008
LIMITED_HISTORY_MIN_BLOCK_ROWS = 42
MIN_TRAIN_ROWS = 60
STRICT_MAX_TRAIN_ROWS = 1260
INNER_MAX_FOLDS = 3
INNER_MIN_TRAIN_ROWS = 126
LIMITED_HISTORY_N_TRIALS = 8


TRANSACTION_COST_BPS = 10
SLIPPAGE_BPS = 5
LONG_THRESHOLD = 0.0
SIGNAL_ANCHOR = "expanding_median"

MLFLOW_ENABLED = os.environ.get("MLFLOW_ENABLED", "true").strip().lower() in {"1", "true", "yes", "y"}
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT_NAME = os.environ.get("MLFLOW_EXPERIMENT_NAME", "stock_return_forecasting_research")
MLFLOW_LOG_OPTUNA_TRIALS = os.environ.get("MLFLOW_LOG_OPTUNA_TRIALS", "true").strip().lower() in {"1", "true", "yes", "y"}

HORIZONS = [
    {"name": "week", "horizon": 5},
    {"name": "month", "horizon": 21},
]

MOMENTUM_CANDIDATE_COLUMNS = [
    "ret_lag_1",
    "ret_lag_2",
    "ret_lag_3",
    "ret_lag_5",
    "ret_lag_10",
    "ret_lag_20",
    "rolling_ret_mean_5",
    "rolling_ret_mean_10",
    "rolling_ret_mean_20",
    "rolling_ret_mean_60",
]


def with_tuning(config: dict) -> dict:
    return {**config, "n_trials": N_TRIALS, "optuna_n_jobs": OPTUNA_N_JOBS}


def make_model_configs(feature_cols: list[str]) -> list[dict]:
    momentum_choices = [col for col in MOMENTUM_CANDIDATE_COLUMNS if col in feature_cols]
    configs = [
        {
            "name": "naive_persistence",
            "model_type": "naive_persistence",
            "estimator_factory": build_model,
            "static_params": {},
            "search_space": {},
            "needs_scaler": False,
        },
        with_tuning(
            {
                "name": "momentum",
                "model_type": "momentum",
                "estimator_factory": build_model,
                "static_params": {},
                "search_space": {"column": {"type": "categorical", "choices": momentum_choices}},
                "needs_scaler": False,
            }
        ),
        with_tuning(
            {
                "name": "ridge",
                "model_type": "ridge",
                "estimator_factory": build_model,
                "static_params": {},
                "search_space": {
                    "alpha": {"type": "float", "low": 1e-3, "high": 1e5, "log": True},
                    "solver": {"type": "categorical", "choices": ["auto", "svd", "cholesky", "lsqr"]},
                    "fit_intercept": {"type": "categorical", "choices": [True, False]},
                    "tol": {"type": "float", "low": 1e-6, "high": 1e-2, "log": True},
                },
                "needs_scaler": True,
            }
        ),
        with_tuning(
            {
                "name": "hist_gradient_boosting",
                "model_type": "hist_gradient_boosting",
                "estimator_factory": build_model,
                "static_params": {},
                "search_space": {
                    "max_iter": {"type": "int", "low": 20, "high": 400},
                    "learning_rate": {"type": "float", "low": 1e-3, "high": 0.1, "log": True},
                    "l2_regularization": {"type": "float", "low": 1e-4, "high": 10.0, "log": True},
                    "max_leaf_nodes": {"type": "int", "low": 3, "high": 31},
                    "max_depth": {"type": "categorical", "choices": [None, 2, 3, 4, 6, 8]},
                    "min_samples_leaf": {"type": "int", "low": 10, "high": 100},
                    "max_bins": {"type": "int", "low": 32, "high": 255},
                },
                "needs_scaler": False,
            }
        ),
    ]

    if importlib.util.find_spec("lightgbm") is not None:
        configs.append(
            with_tuning(
                {
                    "name": "lightgbm",
                    "model_type": "lightgbm",
                    "estimator_factory": build_model,
                    "static_params": {"objective": "regression", "verbosity": -1, "force_col_wise": True, "n_jobs": 1},
                    "search_space": {
                        "n_estimators": {"type": "int", "low": 25, "high": 400},
                        "learning_rate": {"type": "float", "low": 1e-3, "high": 0.1, "log": True},
                        "num_leaves": {"type": "int", "low": 2, "high": 64},
                        "max_depth": {"type": "categorical", "choices": [-1, 2, 3, 4, 6, 8, 10]},
                        "min_child_samples": {"type": "int", "low": 5, "high": 100},
                        "subsample": {"type": "float", "low": 0.5, "high": 1.0},
                        "subsample_freq": {"type": "int", "low": 1, "high": 10},
                        "colsample_bytree": {"type": "float", "low": 0.5, "high": 1.0},
                        "reg_alpha": {"type": "float", "low": 1e-8, "high": 10.0, "log": True},
                        "reg_lambda": {"type": "float", "low": 1e-3, "high": 100.0, "log": True},
                        "min_split_gain": {"type": "float", "low": 0.0, "high": 1.0},
                    },
                    "needs_scaler": False,
                }
            )
        )

    if importlib.util.find_spec("xgboost") is not None:
        configs.append(
            with_tuning(
                {
                    "name": "xgboost",
                    "model_type": "xgboost",
                    "estimator_factory": build_model,
                    "static_params": {"objective": "reg:squarederror", "verbosity": 0, "n_jobs": 1},
                    "search_space": {
                        "n_estimators": {"type": "int", "low": 25, "high": 400},
                        "learning_rate": {"type": "float", "low": 1e-3, "high": 0.1, "log": True},
                        "max_depth": {"type": "int", "low": 1, "high": 8},
                        "min_child_weight": {"type": "float", "low": 0.1, "high": 20.0, "log": True},
                        "subsample": {"type": "float", "low": 0.5, "high": 1.0},
                        "colsample_bytree": {"type": "float", "low": 0.5, "high": 1.0},
                        "reg_alpha": {"type": "float", "low": 1e-8, "high": 10.0, "log": True},
                        "reg_lambda": {"type": "float", "low": 1e-3, "high": 100.0, "log": True},
                        "gamma": {"type": "float", "low": 0.0, "high": 5.0},
                    },
                    "needs_scaler": False,
                }
            )
        )

    if importlib.util.find_spec("catboost") is not None:
        configs.append(
            with_tuning(
                {
                    "name": "catboost",
                    "model_type": "catboost",
                    "estimator_factory": build_model,
                    "static_params": {
                        "loss_function": "RMSE",
                        "verbose": False,
                        "allow_writing_files": False,
                        "thread_count": 1,
                        "grow_policy": "Depthwise",
                    },
                    "search_space": {
                        "iterations": {"type": "int", "low": 25, "high": 400},
                        "learning_rate": {"type": "float", "low": 1e-3, "high": 0.1, "log": True},
                        "depth": {"type": "int", "low": 2, "high": 8},
                        "l2_leaf_reg": {"type": "float", "low": 1e-2, "high": 100.0, "log": True},
                        "random_strength": {"type": "float", "low": 0.0, "high": 10.0},
                        "bagging_temperature": {"type": "float", "low": 0.0, "high": 10.0},
                        "border_count": {"type": "int", "low": 32, "high": 254},
                        "min_data_in_leaf": {"type": "int", "low": 1, "high": 50},
                    },
                    "needs_scaler": False,
                }
            )
        )

    return configs


## Load EDA Artifacts

In [4]:
horizon_inputs = []
base_feature_cols = None

for spec in HORIZONS:
    horizon_name = spec["name"]
    horizon_dir = DATA_DIR / "horizons" / horizon_name
    model_dataset_path = horizon_dir / "model_dataset.parquet"
    feature_columns_path = horizon_dir / "feature_columns.json"

    if horizon_name == "week" and not model_dataset_path.exists() and not model_dataset_path.with_suffix(".csv").exists():
        model_dataset_path = DATA_DIR / "model_dataset.parquet"
        feature_columns_path = DATA_DIR / "feature_columns.json"

    if not model_dataset_path.exists() and not model_dataset_path.with_suffix(".csv").exists():
        raise FileNotFoundError(f"Run notebooks/01_eda.ipynb before this notebook; missing {model_dataset_path}")
    if not feature_columns_path.exists():
        raise FileNotFoundError(f"Missing {feature_columns_path}. Run notebooks/01_eda.ipynb first")

    model_df = load_table(model_dataset_path)
    model_df["date"] = pd.to_datetime(model_df["date"])
    feature_payload = load_json(feature_columns_path)
    horizon_feature_cols = feature_payload["feature_columns"]
    target_col = feature_payload["target_column"]

    if base_feature_cols is None:
        base_feature_cols = horizon_feature_cols
    elif horizon_feature_cols != base_feature_cols:
        raise ValueError(f"Base feature columns differ for horizon {horizon_name}")

    model_configs = make_model_configs(horizon_feature_cols)
    horizon_inputs.append(
        {
            "horizon_name": feature_payload.get("horizon_name", horizon_name),
            "horizon": int(feature_payload.get("horizon", spec["horizon"])),
            "model_df": model_df,
            "feature_cols": horizon_feature_cols,
            "target_col": target_col,
            "model_configs": model_configs,
            "artifact_dir": ARTIFACT_DIR / "horizons" / horizon_name,
        }
    )

    print({
        "horizon": horizon_name,
        "rows": len(model_df),
        "features": len(horizon_feature_cols),
        "target": target_col,
        "models": [model["name"] for model in model_configs],
    })
    display(model_df[["date", "ticker", target_col, *horizon_feature_cols[:6]]].head())


{'horizon': 'week', 'rows': 19407, 'features': 127, 'target': 'target_return_5_next_open', 'models': ['naive_persistence', 'momentum', 'ridge', 'hist_gradient_boosting']}


,date,ticker,target_return_5_next_open,log_close,ret_1,open_close_ret,high_low_range,close_to_high,close_to_low
0,2015-11-09,CBOM,0.013316,1.321756,-0.003992,0.000000,0.000000,0.000000,0.000000
1,2015-11-10,CBOM,0.019908,1.320422,-0.001334,0.004013,0.004005,0.000000,0.004021
2,2015-11-11,CBOM,0.005312,1.319086,-0.001336,0.002677,0.002674,0.000000,0.002681
3,2015-11-12,CBOM,0.010582,1.323088,0.004003,0.000000,0.000000,0.000000,0.000000
4,2015-11-13,CBOM,-0.002649,1.329724,0.006636,0.005305,0.300265,-0.227783,0.005319


{'horizon': 'month', 'rows': 19295, 'features': 127, 'target': 'target_return_21_next_open', 'models': ['naive_persistence', 'momentum', 'ridge', 'hist_gradient_boosting']}


,date,ticker,target_return_21_next_open,log_close,ret_1,open_close_ret,high_low_range,close_to_high,close_to_low
0,2015-11-09,CBOM,0.027761,1.321756,-0.003992,0.000000,0.000000,0.000000,0.000000
1,2015-11-10,CBOM,0.023842,1.320422,-0.001334,0.004013,0.004005,0.000000,0.004021
2,2015-11-11,CBOM,0.021081,1.319086,-0.001336,0.002677,0.002674,0.000000,0.002681
3,2015-11-12,CBOM,0.021053,1.323088,0.004003,0.000000,0.000000,0.000000,0.000000
4,2015-11-13,CBOM,0.011834,1.329724,0.006636,0.005305,0.300265,-0.227783,0.005319


## Run Strict Protocol

In [5]:
horizon_results = {}

for item in horizon_inputs:
    horizon_name = item["horizon_name"]
    horizon = item["horizon"]
    mlflow_config = MLflowRunConfig(
        tracking_uri=MLFLOW_TRACKING_URI,
        experiment_name=MLFLOW_EXPERIMENT_NAME,
        notebook_name="02_table_model_forecasting",
        horizon_name=horizon_name,
        horizon=horizon,
        enabled=MLFLOW_ENABLED,
        log_optuna_trials=MLFLOW_LOG_OPTUNA_TRIALS,
    )
    mlflow_params = {
        "primary_metric": PRIMARY_METRIC,
        "force_retrain": FORCE_RETRAIN,
        "random_state": RANDOM_STATE,
        "n_trials": N_TRIALS,
        "optuna_n_jobs": OPTUNA_N_JOBS,
        "model_count": len(item["model_configs"]),
        "feature_count": len(item["feature_cols"]),
        "target_col": item["target_col"],
        "validation_rows": STRICT_VALIDATION_ROWS,
        "test_rows": STRICT_TEST_ROWS,
        "transaction_cost_bps": TRANSACTION_COST_BPS,
        "slippage_bps": SLIPPAGE_BPS,
        "signal_anchor": SIGNAL_ANCHOR,
    }
    with mlflow_horizon_run(
        mlflow_config,
        params=mlflow_params,
        tags={"training_protocol": "strict_table_models", "training_notebook": "02_table_model_forecasting"},
    ) as mlflow_run:
        result = run_strict_per_ticker_protocol(
            model_df=item["model_df"],
            feature_cols=item["feature_cols"],
            target_col=item["target_col"],
            model_configs=item["model_configs"],
            artifact_dir=item["artifact_dir"],
            force_retrain=FORCE_RETRAIN,
            primary_metric=PRIMARY_METRIC,
            random_state=RANDOM_STATE,
            run_metadata={"horizon_name": horizon_name, "horizon": horizon},
            validation_rows=STRICT_VALIDATION_ROWS,
            test_rows=STRICT_TEST_ROWS,
            mature_min_rows=MATURE_MIN_ROWS,
            limited_history_min_block_rows=LIMITED_HISTORY_MIN_BLOCK_ROWS,
            min_train_rows=MIN_TRAIN_ROWS,
            max_train_rows=STRICT_MAX_TRAIN_ROWS,
            inner_max_folds=INNER_MAX_FOLDS,
            inner_min_train_rows=INNER_MIN_TRAIN_ROWS,
            limited_history_n_trials=LIMITED_HISTORY_N_TRIALS,
            transaction_cost_bps=TRANSACTION_COST_BPS,
            slippage_bps=SLIPPAGE_BPS,
            long_threshold=LONG_THRESHOLD,
            signal_anchor=SIGNAL_ANCHOR,
            mlflow_trial_logger=mlflow_run.trial_logger,
        )
        log_strict_protocol_result(
            result,
            params=mlflow_params,
            tags={"training_protocol": "strict_table_models", "training_notebook": "02_table_model_forecasting"},
        )
    horizon_results[horizon_name] = result

    print(f"=== Strict protocol: {horizon_name} ({horizon} trading days) ===")
    display(result["outer_splits"])
    display(result["validation_model_ranking"])
    display(result["selected_models_by_ticker"])
    display(result["test_prediction_metrics"])
    display(result["test_signal_metrics"])
    display(result["leakage_audit"])

    failed = result["leakage_audit"][~result["leakage_audit"]["passed"]]
    if not failed.empty:
        raise AssertionError(f"Strict protocol leakage audit failed for {horizon_name}: {failed['check'].tolist()}")


🏃 View run week-momentum-trial-1 at: http://localhost:5000/#/experiments/2/runs/2cd60c4fa0dc47268bfed6e1ea71ecb8
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-momentum-trial-0 at: http://localhost:5000/#/experiments/2/runs/d26441ff9b434aed9b98fae171c51181
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-momentum-trial-2 at: http://localhost:5000/#/experiments/2/runs/fc918229409f466992ab1288eea4e90a
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-momentum-trial-3 at: http://localhost:5000/#/experiments/2/runs/41b481552ac14ff3aec285b40d0081e2
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-momentum-trial-4 at: http://localhost:5000/#/experiments/2/runs/37a6350dbcf041ff882506b5d7625143
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-momentum-trial-5 at: http://localhost:5000/#/experiments/2/runs/a05c8eb3aa7041c3b6b076bec344df4d
🧪 View experiment at: 

,ticker,status,split_quality,limited_history,n_obs,n_train_available,n_train,n_refit_available,n_refit,max_train_rows,n_validation,n_test,train_start,train_end,refit_start,refit_end,validation_start,validation_end,test_start,test_end,train_target_end,refit_target_end,validation_target_end,test_target_end,horizon_name,horizon
0,CBOM,ok,mature,False,2758,2506,1260,2632,1260,1260,126,126,2020-10-19,2025-09-02,2021-04-20,2026-01-24,2025-09-03,2026-01-24,2026-01-25,2026-06-07,2025-09-08,2026-01-30,2026-01-30,2026-06-13,week,5
1,MBNK,ok,limited_history,True,537,323,323,430,430,1260,107,107,2024-09-05,2025-10-12,2024-09-05,2026-02-12,2025-10-13,2026-02-12,2026-02-13,2026-06-07,2025-10-18,2026-02-20,2026-02-20,2026-06-13,week,5
2,SBER,ok,mature,False,4629,4377,1260,4503,1260,1260,126,126,2020-10-19,2025-09-02,2021-04-20,2026-01-24,2025-09-03,2026-01-24,2026-01-25,2026-06-07,2025-09-08,2026-01-30,2026-01-30,2026-06-13,week,5
3,SBERP,ok,mature,False,4629,4377,1260,4503,1260,1260,126,126,2020-10-19,2025-09-02,2021-04-20,2026-01-24,2025-09-03,2026-01-24,2026-01-25,2026-06-07,2025-09-08,2026-01-30,2026-01-30,2026-06-13,week,5
4,SVCB,ok,limited_history,True,636,384,384,510,510,1260,126,126,2024-04-27,2025-09-02,2024-04-27,2026-01-24,2025-09-03,2026-01-24,2026-01-25,2026-06-07,2025-09-08,2026-01-30,2026-01-30,2026-06-13,week,5
5,T,ok,mature,False,1633,1381,1260,1507,1260,1260,126,126,2020-09-07,2025-08-25,2021-03-09,2026-01-18,2025-08-26,2026-01-18,2026-01-19,2026-06-07,2025-08-31,2026-01-24,2026-01-24,2026-06-13,week,5
6,VTBR,ok,mature,False,4585,4333,1260,4459,1260,1260,126,126,2020-10-13,2025-09-02,2021-04-14,2026-01-24,2025-09-03,2026-01-24,2026-01-25,2026-06-07,2025-09-08,2026-01-30,2026-01-30,2026-06-13,week,5


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
2,validation,CBOM,naive_persistence,126,0.042964,0.059218,-0.245459,-0.034424,-0.010808,0.555556,week,5,mature,False,1254,1260,6,2506,1260,2020-10-19,2025-08-27,1,True
1,validation,CBOM,momentum,126,0.041241,0.053707,-0.024420,0.022708,0.006116,0.547619,week,5,mature,False,1254,1260,6,2506,1260,2020-10-19,2025-08-27,2,False
0,validation,CBOM,hist_gradient_boosting,126,0.041974,0.054412,-0.051477,0.006890,-0.019915,0.476190,week,5,mature,False,1254,1260,6,2506,1260,2020-10-19,2025-08-27,3,False
3,validation,CBOM,ridge,126,0.057110,0.072247,-0.853772,-0.137970,-0.188691,0.468254,week,5,mature,False,1254,1260,6,2506,1260,2020-10-19,2025-08-27,4,False
7,validation,MBNK,ridge,107,0.094680,0.107117,-11.783746,0.043888,0.053077,0.532710,week,5,limited_history,True,317,323,6,323,1260,2024-09-05,2025-10-06,1,True
6,validation,MBNK,naive_persistence,107,0.026817,0.035281,-0.386853,-0.123250,-0.138239,0.439252,week,5,limited_history,True,317,323,6,323,1260,2024-09-05,2025-10-06,2,False
4,validation,MBNK,hist_gradient_boosting,107,0.024833,0.033190,-0.227314,-0.363205,-0.311212,0.392523,week,5,limited_history,True,317,323,6,323,1260,2024-09-05,2025-10-06,3,False
5,validation,MBNK,momentum,107,0.025908,0.033969,-0.285618,-0.522973,-0.500735,0.355140,week,5,limited_history,True,317,323,6,323,1260,2024-09-05,2025-10-06,4,False
8,validation,SBER,hist_gradient_boosting,126,0.018699,0.025155,-0.201940,-0.011232,0.019777,0.507937,week,5,mature,False,1254,1260,6,4377,1260,2020-10-19,2025-08-27,1,True
11,validation,SBER,ridge,126,0.017621,0.023261,-0.027785,0.097441,0.066805,0.484127,week,5,mature,False,1254,1260,6,4377,1260,2020-10-19,2025-08-27,2,False


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
2,validation,CBOM,naive_persistence,126,0.042964,0.059218,-0.245459,-0.034424,-0.010808,0.555556,week,5,mature,False,1254,1260,6,2506,1260,2020-10-19,2025-08-27,1,True
7,validation,MBNK,ridge,107,0.094680,0.107117,-11.783746,0.043888,0.053077,0.532710,week,5,limited_history,True,317,323,6,323,1260,2024-09-05,2025-10-06,1,True
8,validation,SBER,hist_gradient_boosting,126,0.018699,0.025155,-0.201940,-0.011232,0.019777,0.507937,week,5,mature,False,1254,1260,6,4377,1260,2020-10-19,2025-08-27,1,True
15,validation,SBERP,ridge,126,0.016561,0.022423,0.001090,0.118673,0.125531,0.492063,week,5,mature,False,1254,1260,6,4377,1260,2020-10-19,2025-08-27,1,True
19,validation,SVCB,ridge,126,0.026622,0.035697,-0.013759,-0.066305,-0.080510,0.571429,week,5,limited_history,True,378,384,6,384,1260,2024-04-27,2025-08-27,1,True
23,validation,T,ridge,126,0.023720,0.031852,-0.011754,0.100464,0.022626,0.523810,week,5,mature,False,1254,1260,6,1381,1260,2020-09-07,2025-08-19,1,True
26,validation,VTBR,naive_persistence,126,0.024557,0.031739,-0.368296,-0.120858,-0.042301,0.515873,week,5,mature,False,1254,1260,6,4333,1260,2020-10-13,2025-08-27,1,True


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_refit_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected,validation_directional_accuracy
0,test,CBOM,hist_gradient_boosting,126,0.049230,0.090028,0.009173,0.112775,0.035261,0.428571,week,5,mature,False,1254,1260,6,2632,1260,2021-04-20,2026-01-18,3,False,0.476190
1,test,CBOM,momentum,126,0.045301,0.092775,-0.052203,-0.289441,-0.208165,0.507937,week,5,mature,False,1254,1260,6,2632,1260,2021-04-20,2026-01-18,2,False,0.547619
2,test,CBOM,naive_persistence,126,0.051369,0.103285,-0.304112,-0.063626,-0.098463,0.555556,week,5,mature,False,1254,1260,6,2632,1260,2021-04-20,2026-01-18,1,True,0.555556
3,test,CBOM,ridge,126,0.075575,0.129198,-1.040579,0.229357,0.414899,0.523810,week,5,mature,False,1254,1260,6,2632,1260,2021-04-20,2026-01-18,4,False,0.468254
4,test,MBNK,hist_gradient_boosting,107,0.013947,0.017689,-0.232208,NaN,NaN,0.327103,week,5,limited_history,True,424,430,6,430,1260,2024-09-05,2026-02-06,3,False,0.392523
5,test,MBNK,momentum,107,0.014253,0.017562,-0.214535,-0.007463,-0.072528,0.504673,week,5,limited_history,True,424,430,6,430,1260,2024-09-05,2026-02-06,4,False,0.355140
6,test,MBNK,naive_persistence,107,0.014817,0.019018,-0.424342,0.018761,0.064461,0.560748,week,5,limited_history,True,424,430,6,430,1260,2024-09-05,2026-02-06,2,False,0.439252
7,test,MBNK,ridge,107,0.084036,0.100423,-38.712928,0.127854,0.164091,0.691589,week,5,limited_history,True,424,430,6,430,1260,2024-09-05,2026-02-06,1,True,0.532710
8,test,SBER,hist_gradient_boosting,126,0.012301,0.015035,-1.026027,0.188845,0.158689,0.523810,week,5,mature,False,1254,1260,6,4503,1260,2021-04-20,2026-01-18,1,True,0.507937
9,test,SBER,momentum,126,0.009175,0.011812,-0.250499,0.015127,0.033035,0.507937,week,5,mature,False,1254,1260,6,4503,1260,2021-04-20,2026-01-18,4,False,0.428571


...

,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
0,validation,CBOM,hist_gradient_boosting,126,0.093036,0.113505,-0.048585,0.388619,0.378322,0.777778,month,21,mature,False,1238,1260,22,2490,1260,2020-09-25,2025-07-22,1,True
2,validation,CBOM,naive_persistence,126,0.099125,0.119212,-0.156673,0.041701,0.054734,0.539683,month,21,mature,False,1238,1260,22,2490,1260,2020-09-25,2025-07-22,2,False
1,validation,CBOM,momentum,126,0.097398,0.119820,-0.168507,-0.284121,-0.268013,0.492063,month,21,mature,False,1238,1260,22,2490,1260,2020-09-25,2025-07-22,3,False
3,validation,CBOM,ridge,126,0.146563,0.169979,-1.351587,-0.273428,-0.428385,0.420635,month,21,mature,False,1238,1260,22,2490,1260,2020-09-25,2025-07-22,4,False
7,validation,MBNK,ridge,104,0.060376,0.073583,-3.841647,0.276997,0.014846,0.644231,month,21,limited_history,True,291,313,22,313,1260,2024-09-05,2025-09-08,1,True
5,validation,MBNK,momentum,104,0.038169,0.048485,-1.102135,-0.127077,-0.094170,0.519231,month,21,limited_history,True,291,313,22,313,1260,2024-09-05,2025-09-08,2,False
6,validation,MBNK,naive_persistence,104,0.037827,0.048219,-1.079135,-0.154034,-0.242047,0.480769,month,21,limited_history,True,291,313,22,313,1260,2024-09-05,2025-09-08,3,False
4,validation,MBNK,hist_gradient_boosting,104,0.044877,0.053186,-1.529513,0.379446,0.357186,0.192308,month,21,limited_history,True,291,313,22,313,1260,2024-09-05,2025-09-08,4,False
10,validation,SBER,naive_persistence,126,0.031213,0.038807,-0.149899,-0.051363,-0.087412,0.515873,month,21,mature,False,1238,1260,22,4361,1260,2020-09-25,2025-07-22,1,True
8,validation,SBER,hist_gradient_boosting,126,0.030958,0.039400,-0.185308,0.495317,0.485675,0.476190,month,21,mature,False,1238,1260,22,4361,1260,2020-09-25,2025-07-22,2,False


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
0,validation,CBOM,hist_gradient_boosting,126,0.093036,0.113505,-0.048585,0.388619,0.378322,0.777778,month,21,mature,False,1238,1260,22,2490,1260,2020-09-25,2025-07-22,1,True
7,validation,MBNK,ridge,104,0.060376,0.073583,-3.841647,0.276997,0.014846,0.644231,month,21,limited_history,True,291,313,22,313,1260,2024-09-05,2025-09-08,1,True
10,validation,SBER,naive_persistence,126,0.031213,0.038807,-0.149899,-0.051363,-0.087412,0.515873,month,21,mature,False,1238,1260,22,4361,1260,2020-09-25,2025-07-22,1,True
12,validation,SBERP,hist_gradient_boosting,126,0.028662,0.037981,-0.243323,0.061474,0.189155,0.492063,month,21,mature,False,1238,1260,22,4361,1260,2020-09-25,2025-07-22,1,True
17,validation,SVCB,momentum,124,0.051505,0.073587,-0.159008,0.330616,0.313385,0.580645,month,21,limited_history,True,350,372,22,372,1260,2024-04-27,2025-07-26,1,True
21,validation,T,momentum,126,0.043278,0.051879,0.002100,0.128773,0.148820,0.587302,month,21,mature,False,1238,1260,22,1365,1260,2020-08-14,2025-07-14,1,True
26,validation,VTBR,naive_persistence,126,0.040148,0.051510,-0.130428,-0.036380,-0.007778,0.515873,month,21,mature,False,1238,1260,22,4317,1260,2020-09-21,2025-07-22,1,True


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_refit_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected,validation_directional_accuracy
0,test,CBOM,hist_gradient_boosting,126,0.118204,0.146826,-0.180727,-0.595460,-0.557919,0.333333,month,21,mature,False,1238,1260,22,2616,1260,2021-03-29,2025-12-09,1,True,0.777778
1,test,CBOM,momentum,126,0.106287,0.140397,-0.079583,-0.501606,-0.501432,0.412698,month,21,mature,False,1238,1260,22,2616,1260,2021-03-29,2025-12-09,3,False,0.492063
2,test,CBOM,naive_persistence,126,0.107434,0.149980,-0.231998,-0.167621,-0.061099,0.523810,month,21,mature,False,1238,1260,22,2616,1260,2021-03-29,2025-12-09,2,False,0.539683
3,test,CBOM,ridge,126,0.186402,0.241538,-2.195327,0.167960,0.184780,0.396825,month,21,mature,False,1238,1260,22,2616,1260,2021-03-29,2025-12-09,4,False,0.420635
4,test,MBNK,hist_gradient_boosting,104,0.028938,0.035786,-0.486085,-0.304904,-0.277056,0.576923,month,21,limited_history,True,395,417,22,417,1260,2024-09-05,2026-01-05,4,False,0.192308
5,test,MBNK,momentum,104,0.034279,0.039127,-0.776441,0.023662,0.015982,0.528846,month,21,limited_history,True,395,417,22,417,1260,2024-09-05,2026-01-05,2,False,0.519231
6,test,MBNK,naive_persistence,104,0.033716,0.039425,-0.803652,-0.073353,-0.068746,0.509615,month,21,limited_history,True,395,417,22,417,1260,2024-09-05,2026-01-05,3,False,0.480769
7,test,MBNK,ridge,104,0.061310,0.076957,-5.872322,0.383606,0.249536,0.740385,month,21,limited_history,True,395,417,22,417,1260,2024-09-05,2026-01-05,1,True,0.644231
8,test,SBER,hist_gradient_boosting,126,0.011299,0.014679,0.014608,0.587916,0.584692,0.722222,month,21,mature,False,1238,1260,22,4487,1260,2021-03-29,2025-12-09,2,False,0.476190
9,test,SBER,momentum,126,0.014083,0.018636,-0.588278,0.010913,0.046005,0.539683,month,21,mature,False,1238,1260,22,4487,1260,2021-03-29,2025-12-09,3,False,0.468254


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning
0,-0.010447,-0.020784,0.090047,252.0,-0.230812,-0.495513,-0.131442,-0.158123,0.003023,8,CBOM,hist_gradient_boosting,overlapping_tranches,126,False
1,-0.108146,-0.204596,0.061078,252.0,-3.349759,-4.265798,-0.131553,-1.555239,0.005669,15,CBOM,momentum,overlapping_tranches,126,False
2,0.047283,0.096801,0.075697,252.0,1.278792,2.358569,-0.075795,1.277141,0.029478,78,CBOM,naive_persistence,overlapping_tranches,126,False
3,0.105911,0.223040,0.086624,252.0,2.574794,6.640956,-0.131442,1.696875,0.003401,9,CBOM,ridge,overlapping_tranches,126,False
4,-0.069830,-0.160880,0.019512,252.0,-8.245146,-8.943810,-0.075117,-2.141735,0.003205,7,MBNK,hist_gradient_boosting,overlapping_tranches,104,False
5,-0.055515,-0.129245,0.016308,252.0,-7.925334,-7.904145,-0.061176,-2.112670,0.023810,52,MBNK,momentum,overlapping_tranches,104,False
6,-0.062684,-0.145174,0.016600,252.0,-8.745503,-8.172379,-0.064829,-2.239321,0.024725,54,MBNK,naive_persistence,overlapping_tranches,104,False
7,-0.002301,-0.005566,0.010639,252.0,-0.523213,-0.408001,-0.014310,-0.388994,0.007326,16,MBNK,ridge,overlapping_tranches,104,False
8,0.000000,0.000000,0.000000,252.0,NaN,NaN,0.000000,NaN,0.000000,0,SBER,hist_gradient_boosting,overlapping_tranches,126,False
9,0.032764,0.066602,0.009384,252.0,7.097276,37.184938,-0.001908,34.905982,0.029478,78,SBER,momentum,overlapping_tranches,126,False


,check,passed,details
0,strict outer splits are available,True,split_rows=7
1,validation predictions are available,True,rows=3432
2,test predictions are available,True,rows=3432
3,outer split dates are chronological,True,bad_rows=0
4,train and refit windows respect max_train_rows,True,"train_over_cap=0, refit_over_cap=0"
5,validation predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
6,test predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
7,final refit target dates end before test starts,True,overlap_rows=0
8,final model payloads exist for test predictions,True,missing_models=0


## Legacy Diagnostics

Legacy walk-forward OOF training is intentionally not part of the primary notebook flow. Final reports must use `strict_protocol/reports/test_predictions.parquet`; if legacy OOF diagnostics are needed later, keep them in a separate notebook/section and never feed them into final comparison plots.